# 04 — Classificador de adulteração

O notebook 04 original, arquivado em `notebooks/arquivo/04_classificador_v1_exploratorio.ipynb`, treinou no `train`, escolheu no `val` e avaliou uma vez no `test`, com AUC de 0,545. Aquela escolha usou uma única divisão e deve ser lida como exploratória. Este notebook tenta melhorar o classificador com features que comparam cada componente de texto com o restante da própria página, e escolhe o modelo por validação cruzada no `train` + `val`.

**Regras.** Nenhuma feature usa as anotações. O `test` não é usado para escolher nada. O critério de parada abaixo foi definido antes de olhar qualquer resultado:

- se a melhor AUC média da validação cruzada for menor que 0,60, o `test` não é avaliado, e essa é a conclusão;
- se for 0,60 ou mais, fica a combinação de maior AUC média (desempate: menos features), e o limiar é a média dos limiares de Youden dos folds de treino.

In [1]:
import sys
sys.path.append("../src")

import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from classification import cv_evaluate
from dataset_loader import load_dataset
from evaluation import wilson_score_interval
from features import FEATURE_NAMES, load_or_extract
from features_v2 import V2_NAMES, load_or_extract_v2

# NumPy 2.0 com a BLAS Accelerate (macOS) emite avisos espúrios de divisão por zero em
# multiplicações de matrizes; verificado no notebook 04 original, arquivado em
# `notebooks/arquivo/04_classificador_v1_exploratorio.ipynb` (diferença de 1e-16 contra um
# cálculo sem BLAS).
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

SEED = 42
DATASET_ROOT = "../data/findit2"
LIMITE_AUC = 0.60

## 1. Dados e integridade

O arquivo `X51006619709.png` aparece em `train.txt` e em `val.txt`. A célula abaixo verifica em que pasta ele está e qual rótulo cada CSV lhe atribui.

In [2]:
nome = "X51006619709.png"
for split in ("train", "val", "test"):
    existe = (__import__("pathlib").Path(DATASET_ROOT) / split / nome).exists()
    linha = [l for l in open(f"{DATASET_ROOT}/{split}.txt", encoding="utf-8") if l.startswith(nome)]
    forged = linha[0].split(",")[3] if linha else "-"
    print(f"{split}: arquivo na pasta = {existe} | listado no CSV = {bool(linha)} | forged = {forged}")

amostras = {split: load_dataset(DATASET_ROOT, split) for split in ("train", "val")}
for split, S in amostras.items():
    print(f"{split}: {len(S)} imagens | adulteradas: {sum(s.is_tampered for s in S)}")

train: arquivo na pasta = True | listado no CSV = True | forged = 0
val: arquivo na pasta = False | listado no CSV = True | forged = 1
test: arquivo na pasta = False | listado no CSV = False | forged = -
aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)
aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)
train: 577 imagens | adulteradas: 94
val: 192 imagens | adulteradas: 33


**Interpretação.** O arquivo está só em `train/`. O `train.txt` o rotula como autêntico (`forged = 0`), e o `val.txt` como adulterado (`forged = 1`); os dois trazem uma anotação de adulteração. O loader ignora a entrada do `val`, porque o arquivo não está na pasta, e mantém a do `train`. Por isso, o `val` segue com 192 imagens e o `train` com 577, e essa imagem entra no `train` como autêntica, apesar da anotação. Trata-se provavelmente de um erro de rótulo em uma imagem; nada foi removido do dataset. Na união `train` + `val`, ela aparece uma única vez.

## 2. Novas features

Para cada componente conexo de texto (máscara de Otsu), mede-se um valor na caixa do componente e compara-se com os demais componentes da mesma página pelo z-score robusto, z = (v − mediana) / (1,4826 · MAD). A feature é o maior |z| da página: uma região editada deve destoar do resto do próprio documento, e não de um valor global.

| Feature | Valor por componente | Base no material |
|---|---|---|
| `ela_comp_max_z` | erro médio da ELA (qualidade 90) | DCT e quantização (slides 127, 133, 376-379, 432-440) |
| `ela_comp_frac_out` | fração de componentes com erro da ELA acima de mediana + 3 escalas robustas | idem |
| `noise_comp_max_z` | desvio do resíduo (imagem − mediana 3x3) | filtro da mediana (slides 201, 208-216) |
| `edge_comp_max_z` | magnitude média do gradiente de Sobel | gradiente e Sobel (slides 225-230) |
| `intensity_comp_max_z` | intensidade média dos pixels do componente | histograma (slides 232-236) |

A mediana, o MAD e o z-score não aparecem nos slides. Componentes muito pequenos (pontos), muito altos ou muito largos (linhas de tabela) são descartados. As 7 features do notebook 04 original (arquivado em `notebooks/arquivo/04_classificador_v1_exploratorio.ipynb`) são mantidas para comparação.

In [3]:
tabelas = {}
for split, S in amostras.items():
    v1 = load_or_extract(S, f"../data/features_{split}.csv")
    v2 = load_or_extract_v2(S, f"../data/features_v2_{split}.csv")
    tabelas[split] = v1.merge(v2, on="image", validate="one_to_one")

dados = pd.concat([tabelas["train"], tabelas["val"]], ignore_index=True)
print(f"train + val: {len(dados)} imagens | adulteradas: {int(dados['is_tampered'].sum())} | autênticas: {int((dados['is_tampered'] == 0).sum())}")
print("\nmédia das novas features por classe (train + val):")
print(dados.groupby("is_tampered")[V2_NAMES].mean().rename(index={0: "autêntica", 1: "adulterada"}).T.round(3))

train + val: 769 imagens | adulteradas: 127 | autênticas: 642

média das novas features por classe (train + val):
is_tampered           autêntica  adulterada
ela_comp_max_z            5.491       8.453
ela_comp_frac_out         0.012       0.016
noise_comp_max_z          5.761       5.900
edge_comp_max_z           5.161       4.990
intensity_comp_max_z      7.080       7.002


**Interpretação.** A feature nova que mais difere entre as classes é `ela_comp_max_z`: o maior z-score do erro da ELA por componente tem média de 5,49 nas autênticas e 8,45 nas adulteradas. As demais ficam próximas: `noise_comp_max_z` de 5,76 contra 5,90, `edge_comp_max_z` de 5,16 contra 4,99 e `intensity_comp_max_z` de 7,08 contra 7,00. Comparar cada componente com a própria página, em vez de resumir a imagem inteira, recupera parte do sinal da ELA que se perdia nas features globais.

## 3. Validação cruzada

As três combinações de features, (a) as 7 atuais, (b) as 5 novas e (c) as 12, são avaliadas com dois modelos: regressão logística (`class_weight="balanced"`, com `StandardScaler` no pipeline) e árvore de decisão de profundidade 3. A validação cruzada é estratificada, com 5 folds repetidos 3 vezes (15 folds, seed 42). Em cada fold, o modelo é treinado nos folds de treino, o limiar de Youden é calculado nas probabilidades dos folds de treino, e a AUC, a sensibilidade e a especificidade são medidas no fold de validação.

In [4]:
conjuntos = {"(a) 7 atuais": FEATURE_NAMES, "(b) 5 novas": V2_NAMES, "(c) todas (12)": FEATURE_NAMES + V2_NAMES}
modelos = {
    "logística": lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    "árvore (prof. 3)": lambda: make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=3, random_state=SEED)),
}

y = dados["is_tampered"].to_numpy()
resultados = []
for nome_conj, cols in conjuntos.items():
    for nome_mod, fabrica in modelos.items():
        r = cv_evaluate(fabrica, dados[cols].to_numpy(), y, n_splits=5, n_repeats=3, seed=SEED)
        resultados.append({"features": nome_conj, "modelo": nome_mod, "n_features": len(cols), **r})

tabela_cv = pd.DataFrame(resultados)
print(tabela_cv[["features", "modelo", "auc_mean", "auc_std", "sens_mean", "spec_mean", "threshold"]]
      .rename(columns={"auc_mean": "AUC média", "auc_std": "desvio AUC", "sens_mean": "sensib.", "spec_mean": "especif.", "threshold": "limiar médio"})
      .round(3).to_string(index=False))

      features           modelo  AUC média  desvio AUC  sensib.  especif.  limiar médio
  (a) 7 atuais        logística      0.589       0.042    0.422     0.663         0.526
  (a) 7 atuais árvore (prof. 3)      0.614       0.042    0.311     0.898         0.315
   (b) 5 novas        logística      0.634       0.056    0.557     0.640         0.484
   (b) 5 novas árvore (prof. 3)      0.589       0.053    0.377     0.762         0.210
(c) todas (12)        logística      0.665       0.047    0.577     0.696         0.497
(c) todas (12) árvore (prof. 3)      0.617       0.063    0.379     0.847         0.223


**Interpretação.** A maior AUC média é da regressão logística com as 12 features (0,665 ± 0,047). As 5 novas sozinhas, com a logística, chegam a 0,634, e as 7 atuais, com a logística, a 0,589, abaixo do limite. Os desvios entre folds, de 0,04 a 0,06, são da ordem das diferenças entre as combinações, de modo que a ordem entre as melhores linhas não é muito firme. No limiar médio dos folds, a combinação escolhida tem sensibilidade de 57,7% e especificidade de 69,6%.

## 4. Critério de parada

Aplica-se a regra definida no início, sem alteração.

In [5]:
melhor = tabela_cv.sort_values(["auc_mean", "n_features"], ascending=[False, True]).iloc[0]
PASSA = melhor["auc_mean"] >= LIMITE_AUC
print(f"melhor combinação: {melhor['features']} + {melhor['modelo']} | AUC média = {melhor['auc_mean']:.3f} ± {melhor['auc_std']:.3f}")
print(f"critério (AUC média >= {LIMITE_AUC:.2f}): {'atingido, o test será avaliado' if PASSA else 'NÃO atingido, o test não será avaliado'}")

melhor combinação: (c) todas (12) + logística | AUC média = 0.665 ± 0.047
critério (AUC média >= 0.60): atingido, o test será avaliado


## 5. Avaliação no `test`

Só é executada se o critério de parada for atingido. Nesse caso, o modelo escolhido é treinado no `train` + `val` e aplicado uma única vez ao `test`, com o limiar congelado.

In [6]:
if PASSA:
    test_samples = load_dataset(DATASET_ROOT, "test")
    T = load_or_extract(test_samples, "../data/features_test.csv").merge(
        load_or_extract_v2(test_samples, "../data/features_v2_test.csv"), on="image", validate="one_to_one")
    cols = conjuntos[melhor["features"]]
    modelo = modelos[melhor["modelo"]]().fit(dados[cols].to_numpy(), y)
    prob = modelo.predict_proba(T[cols].to_numpy())[:, 1]
    yt = T["is_tampered"].to_numpy()
    pred = (prob >= melhor["threshold"]).astype(int)
    vn, fp, fn, vp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    sens, espec = vp / (vp + fn), vn / (vn + fp)
    prec = vp / (vp + fp) if vp + fp else float("nan")
    print(pd.DataFrame([[vn, fp], [fn, vp]], index=["autêntica", "adulterada"], columns=["prev. autêntica", "prev. adulterada"]))
    for nome_m, valor, n in (("precisão", prec, vp + fp), ("sensibilidade", sens, vp + fn), ("especificidade", espec, vn + fp)):
        lo, hi = wilson_score_interval(valor, n)
        print(f"{nome_m}: {valor:.1%} (IC 95%: {lo:.1%} a {hi:.1%})")
    print(f"AUC: {roc_auc_score(yt, prob):.3f} | baseline (tudo autêntica): acurácia = {(yt == 0).mean():.1%}")
else:
    print("Critério de parada não atingido: o test não foi avaliado.")

            prev. autêntica  prev. adulterada
autêntica               119                64
adulterada               19                16
precisão: 20.0% (IC 95%: 12.7% a 30.0%)
sensibilidade: 45.7% (IC 95%: 30.5% a 61.8%)
especificidade: 65.0% (IC 95%: 57.9% a 71.6%)
AUC: 0.626 | baseline (tudo autêntica): acurácia = 83.9%


**Interpretação.** No `test`, o modelo detecta 16 das 35 adulteradas (sensibilidade de 45,7%, IC 95%: 30,5% a 61,8%) e acerta 119 das 183 autênticas (especificidade de 65,0%, IC 95%: 57,9% a 71,6%). A precisão é de 20,0% (IC 95%: 12,7% a 30,0%), pouco acima da proporção de adulteradas no `test` (16,1%). A AUC é 0,626, dentro da faixa esperada pela validação cruzada (0,665 ± 0,047). Pela matriz de confusão, a acurácia é de (119 + 16) / 218 = 61,9%, abaixo do baseline que chama tudo de autêntico (83,9%).

## 6. Conclusão

**Resultado.** Com features que comparam cada componente de texto com a própria página, a regressão logística com as 12 features atingiu AUC de 0,665 na validação cruzada (train + val) e 0,626 no `test`, avaliado uma única vez. O notebook 04 original, arquivado em `notebooks/arquivo/04_classificador_v1_exploratorio.ipynb`, que usava só as 7 features globais e uma única divisão para a escolha, teve AUC de 0,545 no `test`; ele deve ser lido como exploratório. A melhora vem principalmente do erro da ELA comparado entre componentes (`ela_comp_max_z`).

**Leitura honesta.** O ganho é real, mas pequeno. No limiar escolhido, o modelo deixa passar mais da metade das adulteradas (19 de 35) e marca 35% das autênticas (64 de 183). A precisão de 20% significa que, de cada cinco imagens sinalizadas, cerca de uma é de fato adulterada. A acurácia fica abaixo da do classificador trivial, porque as adulteradas são só 16% do `test`. O classificador não serve para decidir sozinho se um documento foi adulterado; pode, no máximo, ordenar documentos para revisão humana, com AUC próxima de 0,63.

**Limitações.**
- Com 127 adulteradas em train + val e 35 no `test`, os intervalos de confiança são largos (sensibilidade de 30,5% a 61,8%).
- O máximo de um z-score tende a crescer com o número de componentes da página, então páginas com mais texto podem ter valores maiores mesmo sem adulteração. Esse efeito não foi medido nem corrigido.
- Uma imagem do `train` (`X51006619709.png`) tem rótulo provavelmente errado (autêntica no `train.txt`, adulterada no `val.txt`).
- As features dependem da máscara de Otsu; texto claro ou fundo irregular reduzem a qualidade dos componentes.
